In [ ]:
import os
import re
from dotenv import load_dotenv
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from helpers import get_chunks_fixed_size_with_overlap, stable_int_id
from qmd_extraction import load_qmd_files

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")  # optional
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")

if not QDRANT_URL:
    raise RuntimeError("QDRANT_URL is not set in .env")
if not EMBEDDING_MODEL:
    raise RuntimeError("EMBEDDING_MODEL is not set in .env")

CHUNK_SIZE = 300
CHUNK_OVERLAP = 0.1

# Build docs
quarto_docs = []
for data in load_qmd_files():
    path = data["path"]
    doc_title = path.split("/")[-1]
    chunks = get_chunks_fixed_size_with_overlap(data["content"], CHUNK_SIZE, CHUNK_OVERLAP)

    for i, chunk in enumerate(chunks):
        quarto_docs.append({
            "title": f"{doc_title} - Chunk {i+1}",
            "content": chunk,
            "source": path,
            "chunk": i + 1,
        })

print(f"✅ Built {len(quarto_docs)} chunks")

# Qdrant client
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Embedder
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
dim = embedding_model.get_sentence_embedding_dimension()

# Safe collection name
model_safe = re.sub(r"[^\w]+", "_", EMBEDDING_MODEL)
chunking_method = "custom_overlap_automated"
collection_name = f"QUARTO_Embedding_{model_safe}_Chunking_{chunking_method}"

# Create collection if missing
existing = {c.name for c in client.get_collections().collections}
if collection_name not in existing:
    print(f"== Creating Collection {collection_name} ==")
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )
else:
    print(f"== Collection {collection_name} Already Exists ==")

# Stream embed + upsert
UPSERT_BATCH = 128
ENCODE_BATCH = 64

points = []
for start in tqdm(range(0, len(quarto_docs), ENCODE_BATCH), desc="Embedding & Upserting"):
    batch_docs = quarto_docs[start:start + ENCODE_BATCH]
    batch_texts = [d["content"] for d in batch_docs]

    batch_vecs = embedding_model.encode(
        batch_texts,
        normalize_embeddings=True,
        batch_size=ENCODE_BATCH,
    )

    for doc, vec in zip(batch_docs, batch_vecs):
        pid = stable_int_id(f'{doc["source"]}::{doc["chunk"]}')
        points.append(
            PointStruct(
                id=pid,
                vector=vec.tolist(),
                payload={
                    "title": doc["title"],
                    "content": doc["content"],
                    "source": doc["source"],
                    "chunk": doc["chunk"],
                    "embedding_model": EMBEDDING_MODEL,
                },
            )
        )

    if len(points) >= UPSERT_BATCH:
        client.upsert(collection_name=collection_name, points=points)
        points = []

if points:
    client.upsert(collection_name=collection_name, points=points)

print("✅ Done")

/Users/sofialendahl/Desktop/quarto-wiki-chatbot/chatbot_sofia/scripts/helpers.py:72: SyntaxWarning: invalid escape sequence '\s'
  source_text = re.sub("\s+", " ", source_text)  # Replace multiple whitespces
/Users/sofialendahl/Desktop/quarto-wiki-chatbot/chatbot_sofia/scripts/helpers.py:73: SyntaxWarning: invalid escape sequence '\s'
  return re.split("\s", source_text)  # Split by single whitespace
/var/folders/v1/8f1wz07923l5t5v1d78lxgzm0000gn/T/ipykernel_89004/357978888.py:44: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


✅ Built 6 chunks
== Creating Collection QUARTO_Embedding_all_MiniLM_L6_v2_Chunking_custom_overlap_automated ==


Embedding+Upserting: 100%|██████████| 1/1 [00:00<00:00,  3.79it/s]

✅ Done


In [2]:
quarto_docs

[{'title': 'main.qmd - Chunk 1',
  'content': '--- title: Data Products & Use Cases Overview id: dp-us author: Ostermann, Hampton date: "08/23/2024" date-modified: last-modified --- Technical and non-technical stakeholders outside of People Analytics may not be aware of the analytic value our data process brings. Try sharing your version of the following: "Our People Analytics team has developed automation processes to retrieve and transform data from the Personnel Data Warehouse (PDW) into curated content that is sustainable and AIML ready. The resulting human capital datalake serves as the data source analytics and visualizations, with little to no manual updates required. This approach is different from using BOBJ reports to pull PDW data directly, as it allows for continuous updates while enhancing efficiency and accuracy in the dashboard.""',
  'source': 'site/data-products-use-cases/main.qmd',
  'chunk': 1},
 {'title': 'data-process-and-roles.qmd - Chunk 1',
  'content': '--- tit

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # MiniLM
dim = model.get_sentence_embedding_dimension()   # usually 384



texts = ["hello world", "another sentence"]
embeddings = model.encode(texts, normalize_embeddings=True)  # returns numpy array

In [23]:
client = QdrantClient(url=QDRANT_URL)

from sentence_transformers import SentenceTransformer
embed_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embed_name)  # MiniLM
dim = embedding_model.get_sentence_embedding_dimension()   # usually 384

# Iterate through embedding models
chunking_method = "custom_overlap_automated"  # Clarify chunking - manual
collection_name = f"QUARTO_Embedding_{embed_name}_Chunking_{chunking_method}"

# Check if the collection already exists
existing_collections = {c.name for c in client.get_collections().collections}
if collection_name in existing_collections:
    print(f"== Collection {collection_name} Already Exists ==")
    
# Create a new collection
else:
    print(f"== Creating Collection {collection_name} ==")

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
    )

    # Load Quarto document chunks into Qdrant
    from qdrant_client.models import PointStruct
    import hashlib

    def stable_int_id(s: str) -> int:
        # stable across runs/machines
        return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:16], 16)

    points = []
    batch_size = 128

    texts_to_embed = [doc["content"] for doc in quarto_docs]
    vectors = embedding_model.encode(texts_to_embed, normalize_embeddings=True, batch_size=64)

    for doc, vec in zip(quarto_docs, vectors):
        pid = stable_int_id(f'{doc["source"]}::{doc["title"]}')
        points.append(
            PointStruct(
                id=pid,
                vector=vec.tolist(),
                payload={
                    "title": doc["title"],
                    "content": doc["content"],
                    "source": doc["source"],
                },
            )
        )

        if len(points) >= batch_size:
            client.upsert(collection_name=collection_name, points=points)
            points = []

    if points:
        client.upsert(collection_name=collection_name, points=points)

== Collection QUARTO_Embedding_all-MiniLM-L6-v2_Chunking_custom_overlap_automated Already Exists ==
